### 09 — Interactive Maps Dashboard (Click-to-Inspect)

Goal:
- Show **Clustering archetypes** (latest available from Notebook 08) on a Europe map.
- Add a **fixed right-side details panel** that updates on **click** (not hover).
- Optionally include Regression forecast + Classification risk values in the details panel for context.

Outputs:
- reports/figures/ml_dashboard/ml09_clusters_clickpanel_<quarter>.html
- reports/tables/ml_dashboard/ml09_clusterLegend_latest<clusterQuarter>.csv
- reports/tables/ml_dashboard/ml09_dashboardInputs_<quarter>.parquet


In [6]:
import os, re, json, sys
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

print("Python:", sys.version)
pio.templates.default = "plotly_white"

# open in system browser (separate window)
try:
    pio.renderers.default = "browser" if "browser" in pio.renderers else pio.renderers.default
    print("Plotly renderer:", pio.renderers.default)
except Exception as e:
    print("Renderer setup warning:", repr(e))


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / "README.md").exists() and (p / "data_processed").exists():
            return p
    return start.parent if start.name == "notebooks" else start


projectRoot = find_project_root()
print("projectRoot:", projectRoot)

tablesDir  = projectRoot / "reports" / "tables"
figuresDir = projectRoot / "reports" / "figures"

tablesDashboardDir  = tablesDir / "ml_dashboard"
figuresDashboardDir = figuresDir / "ml_dashboard"
tablesDashboardDir.mkdir(parents=True, exist_ok=True)
figuresDashboardDir.mkdir(parents=True, exist_ok=True)

# IMPORTANT: define these dirs here so nothing depends on previous state
regDir = tablesDir / "ml_regression_inflation_forecast_2025"
clsDir = tablesDir / "ml_classification_high_inflation"
cluDir = tablesDir / "ml_clustering"

print("regDir:", regDir)
print("clsDir:", clsDir)
print("cluDir:", cluDir)

assert regDir.exists(), f"Missing: {regDir}"
assert clsDir.exists(), f"Missing: {clsDir}"
assert cluDir.exists(), f"Missing: {cluDir}"


def pickLatestFile(folder, patterns):
    folder = Path(folder)
    hits = []
    for pat in patterns:
        hits += list(folder.glob(pat))
    hits = [p for p in hits if p.is_file()]
    if not hits:
        raise FileNotFoundError(f"No files matched in {folder} for patterns: {patterns}")
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0]


def to_period_qdec(x):
    if x is None or pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if not s:
        return pd.NaT
    try:
        return pd.Period(s, freq="Q-DEC")
    except Exception:
        return pd.NaT


def period_to_str(x):
    if x is None or x is pd.NaT or pd.isna(x):
        return None
    return str(x)


EXCLUDE_GEOS = {"EA20", "EA19", "EU27_2020", "EU27", "EU28", "EU"}

ISO2_TO_ISO3_FALLBACK = {
    "AT":"AUT","BE":"BEL","BG":"BGR","CH":"CHE","CY":"CYP","CZ":"CZE",
    "DE":"DEU","DK":"DNK","EE":"EST","ES":"ESP","FI":"FIN","FR":"FRA",
    "HR":"HRV","HU":"HUN","IE":"IRL","IS":"ISL","IT":"ITA","LT":"LTU",
    "LU":"LUX","LV":"LVA","MT":"MLT","NL":"NLD","NO":"NOR","PL":"POL",
    "PT":"PRT","RO":"ROU","RS":"SRB","SE":"SWE","SI":"SVN","SK":"SVK",
    "EL":"GRC","UK":"GBR",
}

def iso2_to_iso3(geoCode):
    if geoCode is None:
        return None
    g = str(geoCode).strip().upper()
    if g in EXCLUDE_GEOS:
        return None
    if g in ISO2_TO_ISO3_FALLBACK:
        return ISO2_TO_ISO3_FALLBACK[g]
    if not re.fullmatch(r"[A-Z]{2}", g):
        return None
    try:
        import pycountry
        c = pycountry.countries.get(alpha_2=g)
        return c.alpha_3 if c else None
    except Exception:
        return None


Python: 3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:56:31) [Clang 19.1.7 ]
Plotly renderer: browser
projectRoot: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income
regDir: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_regression_inflation_forecast_2025
clsDir: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_classification_high_inflation
cluDir: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_clustering


In [7]:
import os, re, json, sys
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

print("Python:", sys.version)
pio.templates.default = "plotly_white"

# open in system browser (separate window)
try:
    pio.renderers.default = "browser" if "browser" in pio.renderers else pio.renderers.default
    print("Plotly renderer:", pio.renderers.default)
except Exception as e:
    print("Renderer setup warning:", repr(e))


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / "README.md").exists() and (p / "data_processed").exists():
            return p
    return start.parent if start.name == "notebooks" else start


projectRoot = find_project_root()
print("projectRoot:", projectRoot)

tablesDir  = projectRoot / "reports" / "tables"
figuresDir = projectRoot / "reports" / "figures"

tablesDashboardDir  = tablesDir / "ml_dashboard"
figuresDashboardDir = figuresDir / "ml_dashboard"
tablesDashboardDir.mkdir(parents=True, exist_ok=True)
figuresDashboardDir.mkdir(parents=True, exist_ok=True)

# IMPORTANT: define these dirs here so nothing depends on previous state
regDir = tablesDir / "ml_regression_inflation_forecast_2025"
clsDir = tablesDir / "ml_classification_high_inflation"
cluDir = tablesDir / "ml_clustering"

print("regDir:", regDir)
print("clsDir:", clsDir)
print("cluDir:", cluDir)

assert regDir.exists(), f"Missing: {regDir}"
assert clsDir.exists(), f"Missing: {clsDir}"
assert cluDir.exists(), f"Missing: {cluDir}"


def pickLatestFile(folder, patterns):
    folder = Path(folder)
    hits = []
    for pat in patterns:
        hits += list(folder.glob(pat))
    hits = [p for p in hits if p.is_file()]
    if not hits:
        raise FileNotFoundError(f"No files matched in {folder} for patterns: {patterns}")
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0]


def to_period_qdec(x):
    if x is None or pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if not s:
        return pd.NaT
    try:
        return pd.Period(s, freq="Q-DEC")
    except Exception:
        return pd.NaT


def period_to_str(x):
    if x is None or x is pd.NaT or pd.isna(x):
        return None
    return str(x)


EXCLUDE_GEOS = {"EA20", "EA19", "EU27_2020", "EU27", "EU28", "EU"}

ISO2_TO_ISO3_FALLBACK = {
    "AT":"AUT","BE":"BEL","BG":"BGR","CH":"CHE","CY":"CYP","CZ":"CZE",
    "DE":"DEU","DK":"DNK","EE":"EST","ES":"ESP","FI":"FIN","FR":"FRA",
    "HR":"HRV","HU":"HUN","IE":"IRL","IS":"ISL","IT":"ITA","LT":"LTU",
    "LU":"LUX","LV":"LVA","MT":"MLT","NL":"NLD","NO":"NOR","PL":"POL",
    "PT":"PRT","RO":"ROU","RS":"SRB","SE":"SWE","SI":"SVN","SK":"SVK",
    "EL":"GRC","UK":"GBR",
}

def iso2_to_iso3(geoCode):
    if geoCode is None:
        return None
    g = str(geoCode).strip().upper()
    if g in EXCLUDE_GEOS:
        return None
    if g in ISO2_TO_ISO3_FALLBACK:
        return ISO2_TO_ISO3_FALLBACK[g]
    if not re.fullmatch(r"[A-Z]{2}", g):
        return None
    try:
        import pycountry
        c = pycountry.countries.get(alpha_2=g)
        return c.alpha_3 if c else None
    except Exception:
        return None


Python: 3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:56:31) [Clang 19.1.7 ]
Plotly renderer: browser
projectRoot: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income
regDir: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_regression_inflation_forecast_2025
clsDir: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_classification_high_inflation
cluDir: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_clustering


In [8]:
geoLabelsPath = projectRoot / "metadata" / "geo_labels_36.csv"
assert geoLabelsPath.exists(), f"Missing: {geoLabelsPath}"

dfGeoLabels = pd.read_csv(geoLabelsPath)

if "geo_label" not in dfGeoLabels.columns:
    for alt in ["name", "label", "country"]:
        if alt in dfGeoLabels.columns:
            dfGeoLabels = dfGeoLabels.rename(columns={alt: "geo_label"})
            break

dfGeoLabels = dfGeoLabels.loc[:, ["geo", "geo_label"]].drop_duplicates()
print("dfGeoLabels:", dfGeoLabels.shape)
display(dfGeoLabels.head())


dfGeoLabels: (36, 2)


,geo,geo_label
0,AT,Austria
1,BE,Belgium
2,BG,Bulgaria
3,CH,Switzerland
4,CY,Cyprus


In [9]:
regFiles = sorted(regDir.glob("forecast_2025_*.csv"))
assert len(regFiles) > 0, f"No forecast_2025_*.csv found in {regDir}"

dfs = []
for p in regFiles:
    df = pd.read_csv(p)
    if "timeQuarter" not in df.columns:
        raise ValueError(f"{p.name} missing timeQuarter. cols={df.columns.tolist()}")
    if "yPred" not in df.columns:
        for alt in ["pred", "y_pred", "forecast"]:
            if alt in df.columns:
                df = df.rename(columns={alt: "yPred"})
                break

    modelName = p.stem.replace("forecast_2025_", "")
    df["model"] = modelName
    df["source_key"] = modelName
    df["source_file"] = p.name
    df["timeQuarter"] = df["timeQuarter"].apply(to_period_qdec)

    dfs.append(df.loc[:, ["geo","timeQuarter","yPred","model","source_key","source_file"]])

dfRegForecast = pd.concat(dfs, ignore_index=True)
print("dfRegForecast:", dfRegForecast.shape)
print("Reg keys:", sorted(dfRegForecast["source_key"].unique().tolist()))
display(dfRegForecast.head(3))


dfRegForecast: (496, 6)
Reg keys: ['ENS_M0_plus_MA_Ridge_plus_MB_Ridge', 'ENS_M0_plus_MB_Ridge_weighted', 'M0_baselinePersistence', 'MB_Ridge_recentWindow']


,geo,timeQuarter,yPred,model,source_key,source_file
0,AT,2025Q1,2.266846,ENS_M0_plus_MA_Ridge_plus_MB_Ridge,ENS_M0_plus_MA_Ridge_plus_MB_Ridge,forecast_2025_ENS_M0_plus_MA_Ridge_plus_MB_Rid...
1,AT,2025Q2,2.266846,ENS_M0_plus_MA_Ridge_plus_MB_Ridge,ENS_M0_plus_MA_Ridge_plus_MB_Ridge,forecast_2025_ENS_M0_plus_MA_Ridge_plus_MB_Rid...
2,AT,2025Q3,2.266846,ENS_M0_plus_MA_Ridge_plus_MB_Ridge,ENS_M0_plus_MA_Ridge_plus_MB_Ridge,forecast_2025_ENS_M0_plus_MA_Ridge_plus_MB_Rid...


In [10]:
clsAllPath = clsDir / "cls_forecast_2025_ALL_models.parquet"
assert clsAllPath.exists(), f"Missing: {clsAllPath}"

dfClsAll = pd.read_parquet(clsAllPath)

dfClsAll["timeQuarter"] = dfClsAll["timeQuarter"].apply(to_period_qdec)
print("dfClsAll:", dfClsAll.shape, "| cols:", dfClsAll.columns.tolist())

# prefer models with yProb + 'logit' in name
models_with_prob = []
for m, g in dfClsAll.groupby("model"):
    if g["yProb"].notna().any():
        models_with_prob.append(m)

assert len(models_with_prob) > 0, "No models with yProb found (needed for risk context)."

preferred = [m for m in models_with_prob if "logit" in str(m).lower()]
clsDefaultModel = preferred[0] if preferred else sorted(models_with_prob)[0]

print("Chosen clsDefaultModel:", clsDefaultModel)


dfClsAll: (256, 5) | cols: ['geo', 'timeQuarter', 'yProb', 'yPred', 'model']
Chosen clsDefaultModel: logit_thr_0.010_scenarioFixedX_2025


In [11]:
# locate clustering artifacts
labelsPath         = pickLatestFile(cluDir, patterns=["*labelsWithNames.csv","*labels.csv"])
centroidsShortPath = pickLatestFile(cluDir, patterns=["*centroids_shortCP.csv"])
sigTop4Path        = pickLatestFile(cluDir, patterns=["*clusterCategorySignature_top4.csv"])
clusterMetaPath    = pickLatestFile(cluDir, patterns=["*clusterMeta.csv"])
cpNameMapPath      = pickLatestFile(cluDir, patterns=["*cp01_cp12_name_map.csv"])

dfClusters     = pd.read_csv(labelsPath)
centroidsShort = pd.read_csv(centroidsShortPath)
sigTop4        = pd.read_csv(sigTop4Path)
clusterMeta    = pd.read_csv(clusterMetaPath)
cpNameMap      = pd.read_csv(cpNameMapPath)

print("Loaded clustering files:")
print(" -", Path(labelsPath).name)
print(" -", Path(centroidsShortPath).name)
print(" -", Path(sigTop4Path).name)
print(" -", Path(clusterMetaPath).name)
print(" -", Path(cpNameMapPath).name)

# normalize timeQuarter in cluster labels
if "timeQuarter" not in dfClusters.columns and "timeQuarterPeriod" in dfClusters.columns:
    dfClusters = dfClusters.rename(columns={"timeQuarterPeriod":"timeQuarter"})
dfClusters["timeQuarter"] = dfClusters["timeQuarter"].apply(to_period_qdec)

# cluster id col normalize
if "clusterId" not in dfClusters.columns:
    for alt in ["cluster","kmeansCluster","label"]:
        if alt in dfClusters.columns:
            dfClusters = dfClusters.rename(columns={alt:"clusterId"})
            break
assert "clusterId" in dfClusters.columns, f"dfClusters missing clusterId. cols={dfClusters.columns.tolist()}"

if "clusterId" not in centroidsShort.columns:
    for alt in ["cluster","label","kmeansCluster"]:
        if alt in centroidsShort.columns:
            centroidsShort = centroidsShort.rename(columns={alt:"clusterId"})
            break
assert "clusterId" in centroidsShort.columns, f"centroidsShort missing clusterId. cols={centroidsShort.columns.tolist()}"

# cp name map uses coicopCode/coicopName in your project
assert set(cpNameMap.columns) >= {"coicopCode","coicopName"}, f"Unexpected cpNameMap cols: {cpNameMap.columns.tolist()}"
cp_to_name = dict(zip(cpNameMap["coicopCode"].astype(str), cpNameMap["coicopName"].astype(str)))

def pretty_cp(cp):
    cp = str(cp)
    return f"{cp} ({cp_to_name.get(cp, cp)})"

# CP columns in centroids
cpCols = [c for c in centroidsShort.columns if str(c).startswith("CP") and len(str(c)) == 4]
assert len(cpCols) >= 10, f"Expected CP01..CP12 cols, got: {cpCols}"

# intensity stats
centroidsShort["meanSigned"] = centroidsShort[cpCols].mean(axis=1)
centroidsShort["meanAbs"]    = centroidsShort[cpCols].abs().mean(axis=1)
centroidsShort["maxAbs"]     = centroidsShort[cpCols].abs().max(axis=1)
centroidsShort["topCP"]      = centroidsShort[cpCols].abs().idxmax(axis=1)

# signature column detection
posCol = next((c for c in sigTop4.columns if "toppositive" in str(c).lower()), None)
negCol = next((c for c in sigTop4.columns if "topnegative" in str(c).lower()), None)
assert posCol and negCol, f"Signature columns not found in {sigTop4.columns.tolist()}"
sigTop4 = sigTop4.rename(columns={posCol:"topPositiveCats", negCol:"topNegativeCats"})

# build meaningful names
def cluster_intensity(meanAbs, maxAbs):
    if maxAbs >= 6.0 or meanAbs >= 2.5:
        return "High"
    if maxAbs >= 3.0 or meanAbs >= 1.5:
        return "Moderate"
    return "Mild"

nameMap = {}
for r in centroidsShort.itertuples(index=False):
    cid = int(getattr(r, "clusterId"))
    meanSigned = float(getattr(r, "meanSigned"))
    meanAbs    = float(getattr(r, "meanAbs"))
    maxAbs     = float(getattr(r, "maxAbs"))
    topCP      = str(getattr(r, "topCP"))
    inten      = cluster_intensity(meanAbs, maxAbs)

    if meanSigned < 0:
        nameMap[cid] = f"Below-EU broad (Mild disinflation)"
    else:
        nameMap[cid] = f"Above-EU {inten} ({pretty_cp(topCP)}-led divergence)"

print("Cluster names:")
for k,v in sorted(nameMap.items()):
    print(" ", k, "->", v)

latestClusterQuarter = dfClusters["timeQuarter"].max()
print("latestClusterQuarter:", latestClusterQuarter)

legendDf = (
    clusterMeta.merge(sigTop4[["clusterId","topPositiveCats","topNegativeCats"]], on="clusterId", how="left")
              .merge(centroidsShort[["clusterId","topCP","meanSigned","meanAbs","maxAbs"]], on="clusterId", how="left")
              .sort_values("clusterId")
              .reset_index(drop=True)
)

legendDf["clusterName"] = legendDf["clusterId"].astype(int).map(lambda k: f"{k} — {nameMap.get(k,'Unknown')}")
legendDf["topCP_name"]  = legendDf["topCP"].map(pretty_cp)
legendDf["meanAbs"]     = legendDf["meanAbs"].round(3)
legendDf["maxAbs"]      = legendDf["maxAbs"].round(3)

legendOut = tablesDashboardDir / f"ml09_clusterLegend_latest{latestClusterQuarter}.csv"
legendDf.to_csv(legendOut, index=False)
print("Saved:", legendOut)

display(legendDf)


Loaded clustering files:
 - ml08_coicopClusters_surgeAndAfter_k3_clip01_99_vsEu27_labelsWithNames.csv
 - ml08_coicopClusters_surgeAndAfter_k3_clip01_99_vsEu27_centroids_shortCP.csv
 - ml08_coicopClusters_surgeAndAfter_k3_clip01_99_vsEu27_clusterCategorySignature_top4.csv
 - ml08_coicopClusters_surgeAndAfter_k3_clip01_99_vsEu27_clusterMeta.csv
 - ml08_coicop_cp01_cp12_name_map.csv
Cluster names:
  0 -> Above-EU Mild (CP10 (Education)-led divergence)
  1 -> Below-EU broad (Mild disinflation)
  2 -> Above-EU High (CP04 (Housing, water, electricity, gas & other fuels)-led divergence)
latestClusterQuarter: 2024Q4
Saved: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_dashboard/ml09_clusterLegend_latest2024Q4.csv


,clusterId,clusterLabel,shareOverall,topPositiveCats,topNegativeCats,topCP,meanSigned,meanAbs,maxAbs,clusterName,topCP_name
0,0,"BroadAboveEU (CP10, CP11)",0.308333,CP10 (Education)=2.59; CP11 (Restaurants & hot...,"CP04 (Housing, water, electricity, gas & other...",CP10,1.374307,1.374,2.589,0 — Above-EU Mild (CP10 (Education)-led diverg...,CP10 (Education)
1,1,"BroadBelowEU (CP02, CP01)",0.572917,CP08 (Communication)=-0.28; CP07 (Transport)=-...,CP02 (Alcoholic beverages & tobacco)=-1.70; CP...,CP02,-0.986333,0.986,1.697,1 — Below-EU broad (Mild disinflation),CP02 (Alcoholic beverages & tobacco)
2,2,"BroadAboveEU (CP04, CP11)",0.118750,"CP04 (Housing, water, electricity, gas & other...",CP02 (Alcoholic beverages & tobacco)=2.27; CP0...,CP04,5.324997,5.325,10.992,"2 — Above-EU High (CP04 (Housing, water, elect...","CP04 (Housing, water, electricity, gas & other..."


In [15]:
# =========================
# Cell 7 (FIXED) — Build dashboard table for one quarter
#  - ensures clusterName exists and is NOT overwritten
#  - merges legend info WITHOUT bringing legendDf['clusterName']
# =========================

chosenQuarterStr = "2025Q1"
chosenQuarter = pd.Period(chosenQuarterStr, freq="Q-DEC")

# pick regression key
preferredRegKey = "ENS_M0_plus_MB_Ridge_weighted"
regKeys = sorted(dfRegForecast["source_key"].unique().tolist())
regSourceKey = preferredRegKey if preferredRegKey in regKeys else regKeys[0]
print("regSourceKey:", regSourceKey)

dfRegQ = (
    dfRegForecast[(dfRegForecast["timeQuarter"] == chosenQuarter) & (dfRegForecast["source_key"] == regSourceKey)]
    .loc[:, ["geo","timeQuarter","yPred","model","source_key"]]
    .rename(columns={"yPred":"yPredReg","model":"regModel","source_key":"regSourceKey"})
    .copy()
)

dfClsQ = (
    dfClsAll[(dfClsAll["timeQuarter"] == chosenQuarter) & (dfClsAll["model"] == clsDefaultModel)]
    .loc[:, ["geo","timeQuarter","yProb","yPred","model"]]
    .rename(columns={"yPred":"yPredClass","model":"clsModel"})
    .copy()
)

# latest clusters per geo
dfClusterLatest = (
    dfClusters[dfClusters["timeQuarter"] == latestClusterQuarter]
    .loc[:, ["geo","clusterId"]]
    .rename(columns={"clusterId":"clusterIdLatest"})
    .copy()
)

dfDashQ = (
    dfClsQ.merge(dfRegQ, on=["geo","timeQuarter"], how="left", validate="1:1")
         .merge(dfClusterLatest, on="geo", how="left", validate="m:1")
         .merge(dfGeoLabels, on="geo", how="left", validate="m:1")
)

# map ISO
dfDashQ["iso3"] = dfDashQ["geo"].apply(iso2_to_iso3)
unmapped = sorted(dfDashQ.loc[dfDashQ["iso3"].isna(), "geo"].dropna().unique().tolist())
print("Unmapped excluded:", unmapped)

dfDashQMap = dfDashQ[dfDashQ["iso3"].notna()].copy()

# IMPORTANT: normalize cluster id to Int for stable mapping/joins
dfDashQMap["clusterIdLatestInt"] = pd.to_numeric(dfDashQMap["clusterIdLatest"], errors="coerce").astype("Int64")

# Build clusterName (this is the ONLY clusterName we use)
dfDashQMap["clusterName"] = dfDashQMap["clusterIdLatestInt"].apply(
    lambda v: f"{int(v)} — {nameMap.get(int(v),'Unknown')}" if pd.notna(v) else "Unknown"
)

# Merge legend details WITHOUT legendDf["clusterName"] to avoid collisions
legendCols = ["clusterId","topPositiveCats","topNegativeCats","meanAbs","maxAbs","topCP_name"]
dfDashQMap = dfDashQMap.merge(
    legendDf[legendCols],
    left_on="clusterIdLatestInt",
    right_on="clusterId",
    how="left"
)

# plotly-safe quarter string
dfDashQMap["timeQuarterStr"] = dfDashQMap["timeQuarter"].apply(period_to_str)

# readable risk
dfDashQMap["riskPct"] = 100.0 * pd.to_numeric(dfDashQMap["yProb"], errors="coerce")
dfDashQMap["riskFlag"] = np.where(dfDashQMap["riskPct"].fillna(0.0) >= 1.0, "YES", "no")

outParq = tablesDashboardDir / f"ml09_dashboardInputs_{chosenQuarterStr}.parquet"
dfDashQMap.to_parquet(outParq, index=False)
print("Saved:", outParq)

display(dfDashQMap[["geo","geo_label","clusterIdLatestInt","clusterName","riskPct","yPredReg"]].head(10))


regSourceKey: ENS_M0_plus_MB_Ridge_weighted
Unmapped excluded: ['EA20', 'EU27_2020']
Saved: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/tables/ml_dashboard/ml09_dashboardInputs_2025Q1.parquet


,geo,geo_label,clusterIdLatestInt,clusterName,riskPct,yPredReg
0,AT,Austria,1,1 — Below-EU broad (Mild disinflation),0.019064,2.270636
1,BE,Belgium,0,0 — Above-EU Mild (CP10 (Education)-led diverg...,16.232117,4.816233
2,BG,Bulgaria,1,1 — Below-EU broad (Mild disinflation),0.042977,3.029885
3,CH,Switzerland,1,1 — Below-EU broad (Mild disinflation),0.014506,0.082865
4,CY,Cyprus,1,1 — Below-EU broad (Mild disinflation),0.098065,1.926195
5,CZ,Czechia,0,0 — Above-EU Mild (CP10 (Education)-led diverg...,0.065623,3.528830
6,DE,Germany,0,0 — Above-EU Mild (CP10 (Education)-led diverg...,0.071499,2.398699
7,DK,Denmark,1,1 — Below-EU broad (Mild disinflation),0.016238,1.081507
8,EE,Estonia,0,0 — Above-EU Mild (CP10 (Education)-led diverg...,0.168457,5.288571
9,EL,Greece,1,1 — Below-EU broad (Mild disinflation),0.314770,2.389704


In [17]:
# =========================
# Cell 8 (FIXED) — Full-screen click panel HTML (custom_data aligned per trace)
# =========================
import webbrowser

dfM = dfDashQMap.copy()

# sanity: must be 1 row per country
dups = dfM["iso3"].duplicated().sum()
assert dups == 0, f"Duplicate iso3 rows found ({dups}). You have multiple rows per country."

# Stable discrete colors by clusterId
cluster_ids = sorted([int(x) for x in dfM["clusterIdLatestInt"].dropna().unique().tolist()])
palette = ["#E76F51", "#2A9D8F", "#457B9D", "#8D99AE", "#F4A261"]
cluster_color_map = {}
for i, cid in enumerate(cluster_ids):
    cname = f"{cid} — {nameMap.get(cid,'Unknown')}"
    cluster_color_map[cname] = palette[i % len(palette)]

dfM["clusterName"] = dfM["clusterName"].fillna("Unknown").astype(str)

# columns we want in click panel (order matters!)
custom_cols = [
    "geo_label","geo","timeQuarterStr","clusterName","topCP_name",
    "yPredReg","riskPct","riskFlag",
    "topPositiveCats","topNegativeCats","meanAbs","maxAbs"
]
for c in custom_cols:
    if c not in dfM.columns:
        dfM[c] = None

hovertemplate = "<b>%{customdata[0]}</b><br><span style='color:#444'>Click for details →</span><extra></extra>"

fig = px.choropleth(
    dfM,
    locations="iso3",
    color="clusterName",
    color_discrete_map=cluster_color_map,
    title=f"Clusters (latest={latestClusterQuarter}) shown on {chosenQuarterStr} — click a country for details",
    hover_name="geo_label",
    custom_data=custom_cols,   # ✅ CRITICAL FIX (keeps per-trace alignment)
)

fig.update_layout(margin=dict(l=0,r=0,t=60,b=0), showlegend=False)
fig.update_geos(scope="europe", fitbounds="locations", showcountries=True, showcoastlines=False, showland=True)

# ✅ only set hovertemplate/labels now — DO NOT set customdata here
fig.update_traces(
    hovertemplate=hovertemplate,
    hoverlabel=dict(bgcolor="white", font_color="black", font_size=14, bordercolor="black"),
)

# custom legend html
legend_items_html = ""
for cname, col in cluster_color_map.items():
    legend_items_html += f"""
      <div style="display:flex; align-items:center; gap:10px; margin:6px 0;">
        <div style="width:14px; height:14px; border-radius:3px; background:{col}; border:1px solid #ddd;"></div>
        <div style="font-size:13px; color:#111827;">{cname}</div>
      </div>
    """

div_id = "mapDiv"
map_div = fig.to_html(full_html=False, include_plotlyjs=False, div_id=div_id)

html = f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>Euro Macro — Clusters Click Panel ({chosenQuarterStr})</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    html, body {{ height: 100%; width: 100%; margin: 0; background:#fff; font-family: Arial, sans-serif; }}
    .wrap {{ display:flex; height:100vh; width:100vw; }}
    .left {{ flex: 1 1 auto; min-width: 0; display:flex; }}
    #{div_id} {{ width: 100%; height: 100%; }}
    .right {{ flex: 0 0 420px; border-left: 1px solid #eee; padding: 14px; overflow-y:auto; background:#fff; }}
    .h1 {{ font-size: 14px; font-weight: 800; color:#111827; margin-bottom: 6px; }}
    .sub {{ font-size: 12px; color:#6b7280; margin-bottom: 12px; }}
    .card {{ border:1px solid #e5e7eb; border-radius:10px; padding:12px; background:#fff; box-shadow:0 1px 2px rgba(0,0,0,0.04); margin-top:12px; }}
    .k {{ color:#6b7280; font-size:12px; margin-top:10px; }}
    .v {{ color:#111827; font-size:14px; font-weight:700; margin-top:2px; }}
    .mono {{ font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace; }}
  </style>
</head>

<body>
<div class="wrap">
  <div class="left">{map_div}</div>
  <div class="right">
    <div class="h1">Archetype legend</div>
    <div class="sub">Discrete clusters from Notebook 08 (latest quarter).</div>
    <div class="card">{legend_items_html}</div>

    <div class="h1" style="margin-top:16px;">Country details (click)</div>
    <div class="sub">Hover is minimal so small countries remain clickable.</div>

    <div id="details" class="card">
      <div class="k">Status</div>
      <div class="v">No country selected yet.</div>
    </div>
  </div>
</div>

<script>
  const gd = document.getElementById("{div_id}");
  const details = document.getElementById("details");

  function fmt(x, digits=2) {{
    if (x === null || x === undefined || x === "" || (typeof x === "number" && !isFinite(x))) return "—";
    if (typeof x === "number") return x.toFixed(digits);
    return String(x);
  }}

  function resizePlot() {{
    try {{ Plotly.Plots.resize(gd); }} catch(e) {{}}
  }}
  window.addEventListener("resize", resizePlot);
  setTimeout(resizePlot, 100);
  setTimeout(resizePlot, 400);

  gd.on('plotly_click', function(ev) {{
    if (!ev || !ev.points || !ev.points.length) return;
    const cd = ev.points[0].customdata;

    const geoLabel = cd[0], geo = cd[1], qtr = cd[2], cluster = cd[3], driver = cd[4];
    const yPredReg = cd[5], riskPct = cd[6], riskFlag = cd[7];
    const topPos = cd[8], topNeg = cd[9], meanAbs = cd[10], maxAbs = cd[11];

    details.innerHTML = `
      <div class="k">Country</div>
      <div class="v">${{fmt(geoLabel,0)}} <span class="mono">(${{fmt(geo,0)}})</span></div>

      <div class="k">Forecast quarter</div>
      <div class="v">${{fmt(qtr,0)}}</div>

      <div class="k">Archetype cluster</div>
      <div class="v">${{fmt(cluster,0)}}</div>

      <div class="k">Main driver (centroid)</div>
      <div class="v">${{fmt(driver,0)}}</div>

      <div class="k">Regression forecast (inflation)</div>
      <div class="v">${{fmt(Number(yPredReg),2)}}</div>

      <div class="k">Classification risk (yProb)</div>
      <div class="v">${{fmt(Number(riskPct),3)}}% <span class="mono">flag=${{fmt(riskFlag,0)}}</span></div>

      <div class="k">Signature vs EU27</div>
      <div class="v"><span class="mono">Top +</span>: ${{fmt(topPos,0)}}</div>
      <div class="v"><span class="mono">Top −</span>: ${{fmt(topNeg,0)}}</div>

      <div class="k">Intensity stats</div>
      <div class="v">Mean |Δ|: ${{fmt(Number(meanAbs),3)}} pp</div>
      <div class="v">Max  |Δ|: ${{fmt(Number(maxAbs),3)}} pp</div>
    `;
  }});
</script>

</body>
</html>
"""

outHtml = figuresDashboardDir / f"ml09_clusters_clickpanel_{chosenQuarterStr}.html"
outHtml.write_text(html, encoding="utf-8")
print("Saved:", outHtml)

webbrowser.open(outHtml.as_uri())


Saved: /Users/princebhanusteta/Documents/Projects/euro-macro-inflation-income/reports/figures/ml_dashboard/ml09_clusters_clickpanel_2025Q1.html


True

In [18]:
print("Number of traces:", len(fig.data))
for i,t in enumerate(fig.data):
    print(i, t.name, len(t.locations))


Number of traces: 3
0 1 — Below-EU broad (Mild disinflation) 18
1 0 — Above-EU Mild (CP10 (Education)-led divergence) 11
2 2 — Above-EU High (CP04 (Housing, water, electricity, gas & other fuels)-led divergence) 1
